# Step 0 — Constitutional Typology Pipeline\n",
    "\n",
    "End-to-end pipeline for generating `ccpc_typology_v4.json` from raw inputs.\n",
    "\n",
    "**Stages:**\n",
    "1. Merge 14 mapping JSONs → `ccp_mapping_combined.json`\n",
    "2. Group variables by dimension → `ccp_mapping_by_dimension.json`\n",
    "3. LLM assigns weights & value maps\n",
    "4. Parse & save → `typology/ccpc_typology_v4.json`

In [ ]:
import json, glob, re, os
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_dartmouth.llms import ChatDartmouthCloud

## Stage 2 — Merge 14 Mapping JSONs

In [ ]:
COMBINED_PATH = 'data/ccp_mappings/ccp_mapping_combined.json'

part_files = sorted(glob.glob('data/ccp_mappings/ccp_mapping_part*.json'))
print(f'Found {len(part_files)} part files')

combined = {}
for path in part_files:
    with open(path) as f:
        part = json.load(f)
    section  = part.get('section_label', '')
    part_num = part.get('part', '?')
    for var in part.get('variables', []):
        key = var['variable_name'].lower()
        var['_part']    = part_num
        var['_section'] = section
        combined[key]   = var

with open(COMBINED_PATH, 'w', encoding='utf-8') as f:
    json.dump(combined, f, indent=2)

print(f'Total unique variables: {len(combined)}')
print(f'Saved → {COMBINED_PATH}')

## Stage 2b — Expand Multi-Select Variables into Sub-Variables
80 CCPCNC variables use a multi-select "check all that apply" format. Their parent column
(e.g. `amndprop`) doesn't exist in the dataset — only the binary sub-columns do
(`amndprop_1` … `amndprop_10`). This cell replaces each parent entry with one entry per
sub-variable so Stage 4 can assign per-sub-variable weights and 0/1 value maps.

In [ ]:
# Stage 2b — Expand multi-select parent variables into binary sub-variable entries.
# Multi-select CCPCNC variables (e.g. AMNDPROP, EQUALGR) don't exist as columns
# in the dataset — only their numbered sub-variables do (amndprop_1, amndprop_2, …).
# Replacing each parent with its sub-vars lets the LLM (Stage 4) assign per-sub-var
# weights and value_maps against the actual dataset columns.

SKIP_SUFFIXES = {'90', '96', '97', '98', '99'}

def _expand_subvars(combined):
    to_remove, to_add = [], {}
    for key, var in combined.items():
        cs = var.get('coding_scheme', '')
        if not isinstance(cs, str) or 'Sub-variables' not in cs:
            continue
        entries = re.findall(r'([A-Z][A-Z0-9_]+_\d+)=([^;]+)', cs)
        sub_vars = [
            (name.lower(), label.strip())
            for name, label in entries
            if not any(name.upper().endswith('_' + s) for s in SKIP_SUFFIXES)
        ]
        if not sub_vars:
            continue
        for sv_name, sv_label in sub_vars:
            to_add[sv_name] = {
                'variable_name': sv_name,
                'description':   f'{var.get("description", "")} — {sv_label}',
                'coding_scheme': '0=not selected, 1=selected (99=not applicable)',
                'dimensions':    var.get('dimensions', []),
                'notes':         var.get('notes', ''),
                '_part':         var.get('_part', '?'),
                '_section':      var.get('_section', ''),
                '_parent':       key,
            }
        to_remove.append(key)
    for k in to_remove:
        del combined[k]
    combined.update(to_add)
    return len(to_remove), len(to_add)

orig_count         = len(combined)
n_parents, n_subs  = _expand_subvars(combined)

print(f'Expanded {n_parents} multi-select parents → {n_subs} binary sub-variable entries')
print(f'Combined dict: {orig_count} → {len(combined)} variables')

# Spot-check: show amndprop expansion
sample_subs = sorted(k for k in combined if k.startswith('amndprop_'))
print(f'\namndprop → {len(sample_subs)} sub-vars:')
for s in sample_subs:
    print(f'  {s}: {combined[s]["description"].split(" — ")[-1]}')

with open(COMBINED_PATH, 'w', encoding='utf-8') as f:
    json.dump(combined, f, indent=2)
print(f'\nUpdated → {COMBINED_PATH}')

## Stage 3 — Group by Dimension
Variables belonging to multiple dimensions are duplicated into each.

In [ ]:
DIM_PATH = 'data/ccp_mappings/ccp_mapping_by_dimension.json'

by_dimension = defaultdict(list)
for var in combined.values():
    for dim in var.get('dimensions', []):
        by_dimension[dim].append(var)

by_dimension = {
    dim: sorted(vars_, key=lambda v: v['variable_name'])
    for dim, vars_ in sorted(by_dimension.items())
}

with open(DIM_PATH, 'w', encoding='utf-8') as f:
    json.dump(by_dimension, f, indent=2)

total_entries = sum(len(v) for v in by_dimension.values())
print(f'Dimensions: {len(by_dimension)}')
print()
print(f'{"Dimension":<45} Variables')
print('-' * 55)
for dim, vars_ in by_dimension.items():
    print(f'{dim:<45} {len(vars_)}')
print('-' * 55)
print(f'{"TOTAL (with duplicates)":<45} {total_entries}')
print(f'{"UNIQUE":<45} {len(combined)}')
print()
print(f'Saved → {DIM_PATH}')

## Stage 4 — Parallel LLM Calls (one per dimension)\nEach of the 14 dimensions is scored simultaneously — one LLM call per dimension.

In [ ]:
key = "sk-090de58ca1a646b688dfde94e78abfc4"

def make_llm():
    """Each thread gets its own LLM client instance."""
    return ChatDartmouthCloud(
        model_name="openai.gpt-oss-120b",
        dartmouth_chat_api_key=key,
        streaming=False,
        temperature=0.0,
    )

def build_prompt(dim_name, dim_variables):
    return f'''You are an expert in comparative constitutional law and democratic theory.

Below is a list of constitutional variables from the Comparative Constitutions Project (CCPCNC v5)
that belong to the dimension: {dim_name.upper()}.
Read every variable carefully before responding.

=== VARIABLES ===
{json.dumps(dim_variables, indent=2)}
=== END VARIABLES ===

Your task — for EVERY variable above, produce:

1. VARIABLE WEIGHT — how central this variable is to the {dim_name} dimension.
   Scale:
     0.5 = tangentially related
     1.0 = clearly relevant
     1.5 = important indicator
     2.0 = core measure
     3.0 = critical / definitional
   Base weights on constitutional theory, not data availability.

2. VALUE MAP — for each raw answer code in coding_scheme, a score in [0.0, 1.0] where:
     1.0 = most democratic / rights-protective / constitutionally robust
     0.0 = least democratic / most authoritarian
   Skip codes 96, 97, 98, 99 (missing/inapplicable).
   For 0/1 sub-variables, score 1=selected as positive or negative based on what it implies.
   Use intermediate values (0.25, 0.5, 0.75) for ordinal variables — not just 0 and 1.

3. DIRECTION NOTE — one sentence explaining the scoring direction if at all ambiguous.

Variable names must exactly match the lowercase variable_name values above.

Return ONLY valid JSON, no markdown, no explanation:
{{
  "variable_name": {{
    "weight": <float>,
    "direction_note": "<one sentence>",
    "value_map": {{"<raw_code>": <float 0.0–1.0>}}
  }}
}}'''

In [ ]:
def score_dimension(dim_name, dim_variables):
    """Calls the LLM for a single dimension and returns (dim_name, raw_response)."""
    llm = make_llm()
    prompt = build_prompt(dim_name, dim_variables)
    response = llm.invoke(prompt, max_tokens=1600000000)
    return dim_name, response.content

# ── Fire all 14 dimensions in parallel ────────────────────────────────────────
raw_results = {}   # {dim_name: raw_response}
failed      = []

print(f'Launching {len(by_dimension)} parallel LLM calls...')

with ThreadPoolExecutor(max_workers=14) as executor:
    futures = {
        executor.submit(score_dimension, dim, vars_): dim
        for dim, vars_ in by_dimension.items()
    }
    for future in as_completed(futures):
        dim = futures[future]
        try:
            dim_name, raw = future.result()
            raw_results[dim_name] = raw
            print(f'  ✓  {dim_name} ({len(raw):,} chars)')
        except Exception as e:
            failed.append(dim)
            print(f'  ✗  {dim}: {e}')

print(f'\nDone. {len(raw_results)}/14 succeeded, {len(failed)} failed.')
if failed:
    print(f'Failed dimensions: {failed}')

In [ ]:
# ── Retry failed dimensions ───────────────────────────────────────────────────
# Reads the `failed` list populated by the cell above and retries each dimension
# up to MAX_RETRIES times, with a short backoff between attempts.
# Successful retries are merged into raw_results and removed from failed.

import time

MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds between attempts

if not failed:
    print('No failed dimensions — nothing to retry.')
else:
    print(f'Retrying {len(failed)} failed dimension(s): {failed}')
    still_failed = []

    for dim in failed:
        dim_vars = by_dimension[dim]
        success = False

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                print(
                    f'  [{attempt}/{MAX_RETRIES}] {dim} ...',
                    end=' ',
                    flush=True
                )

                _, raw = score_dimension(dim, dim_vars)

                raw_results[dim] = raw
                print(f'✓ ({len(raw):,} chars)')

                success = True
                break

            except Exception as e:
                print(f'✗ {e}')

                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY)

        if not success:
            still_failed.append(dim)

    # Update the original list in-place so downstream cells see the changes
    failed[:] = still_failed

    print(
        f'\nAfter retries: {len(raw_results)}/14 succeeded, '
        f'{len(failed)} still failed.'
    )

    if failed:
        print(f'Still failed: {failed}')
        print('These dimensions will be missing from the typology.')

## Stage 5 — Parse & Save Typology

In [ ]:
def parse_json_response(raw):
    clean = re.sub(r'^```(?:json)?\s*', '', raw.strip(), flags=re.MULTILINE)
    clean = re.sub(r'```\s*$', '', clean.strip())
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', clean, re.DOTALL)
        return json.loads(m.group())

# Parse each dimension's raw response
typology_by_dim = {}
for dim, raw in raw_results.items():
    try:
        typology_by_dim[dim] = parse_json_response(raw)
        print(f'  ✓  {dim}: {len(typology_by_dim[dim])} variables parsed')
    except Exception as e:
        print(f'  ✗  {dim}: parse error — {e}')

print(f'\nParsed {len(typology_by_dim)}/14 dimensions')

In [ ]:
# Restructure: per-variable format ready for downstream scoring
# { variable_name: { dimensions: {dim: weight}, value_map: {code: score}, direction_note: str } }

typology = {}
for dim, variables in typology_by_dim.items():
    for var_name, var_info in variables.items():
        key = var_name.lower()
        if key not in typology:
            typology[key] = {
                'dimensions':     {},
                'value_map':      var_info.get('value_map', {}),
                'direction_note': var_info.get('direction_note', ''),
            }
        typology[key]['dimensions'][dim] = var_info.get('weight', 1.0)

print(f'Total unique variables in typology: {len(typology)}')
print()
print('Sample entries:')
for k in list(typology.keys())[:3]:
    print(f'  {k}:')
    print(f'    dimensions:  {typology[k]["dimensions"]}')
    print(f'    value_map:   {typology[k]["value_map"]}')
    print(f'    direction:   {typology[k]["direction_note"][:80]}')

In [ ]:
TYPOLOGY_PATH = 'typology/ccpc_typology_v4.json'
os.makedirs('typology', exist_ok=True)

with open(TYPOLOGY_PATH, 'w', encoding='utf-8') as f:
    json.dump(typology, f, indent=2)

print(f'Saved → {TYPOLOGY_PATH}')